# Embedding Similarity Search with Rasteret

Find grain silos across Franklin County, Kansas using
[AlphaEarth Foundations Satellite Embedding dataset (AEF)](https://source.coop/tge-labs/aef)
produced by Google and Google DeepMind — 64-band int8 COGs at 10 m resolution.

This workflow is inspired by:
- Ujaval Gandhi’s GeoPython Tutorials post:
  [Large-Scale Embedding Similarity Search with xarray and Dask](https://www.geopythontutorials.com/notebooks/xarray_embeddings_similarity_search.html)
- Google Earth Engine community tutorial:
  [Satellite Embedding Similarity Search](https://developers.google.com/earth-engine/tutorials/community/satellite-embedding-05-similarity-search)

Data credits:
- County boundary data: **US Census Bureau, 2021 Cartographic Boundary Files**.
- Embeddings: **AlphaEarth Foundations Satellite Embedding dataset (produced by Google and Google DeepMind)**.

| Step | Rasteret API | What it does |
|---|---|---|
| Reference vector | `sample_points` | Extract embeddings at known grain-silo locations |
| Approach A | `get_xarray` | Dense mosaic, cosine similarity |
| Approach B | `get_gdf` | Per-record arrays, cosine similarity |
| Approach C | `to_torchgeo_dataset` | Streaming 1024 px chips, bounded memory |


Attribution:

> "The AlphaEarth Foundations Satellite Embedding dataset is produced by Google and Google DeepMind."


---
## Setup

AEF stores embeddings as int8 with nodata = -128. The dequantization
formula recovers float32 values: `(v / 127.5)^2 * sign(v)`.

All three approaches share the same similarity math — `cosine_similarity_map`
for NumPy and `cosine_similarity_map_torch` for PyTorch chips. Each takes
a `(C, H, W)` cube of raw int8 values and a dequantized reference vector,
and returns a `(H, W)` similarity map.

In [1]:
import time

import duckdb
import geopandas as gpd
import numpy as np
from shapely.geometry import Point
from sklearn.metrics.pairwise import cosine_similarity

import rasteret

rasteret.set_options(progress=False)

ALL_BANDS = [f"A{i:02d}" for i in range(64)]
NODATA = -128

REFERENCE_POINTS = [
    (-95.18616479385629, 38.54715519758577),
    (-95.34468619878159, 38.59339901996762),
    (-95.34280239688128, 38.56233059960432),
]


def dequantize(v: np.ndarray) -> np.ndarray:
    """AEF int8 -> float32. NODATA (-128) becomes NaN."""
    f = v.astype(np.float32)
    nd = (f == NODATA) | np.isnan(f)
    out = (f / 127.5) ** 2 * np.sign(f)
    out[nd] = np.nan
    return out


def cosine_similarity_map(cube: np.ndarray, ref: np.ndarray) -> np.ndarray:
    """Cosine similarity between each pixel and reference embedding via sklearn.

    cube : (C, H, W) int8 array — raw AEF embeddings
    ref  : (C,) float32 reference embedding (already dequantized)

    Returns (H, W) float32 array, NaN where any band is nodata or all-zero.
    """
    C, H, W = cube.shape
    flat = dequantize(cube.reshape(C, -1).T)  # (N, C)

    sim_flat = np.full(flat.shape[0], np.nan, dtype=np.float32)
    valid = np.isfinite(flat).all(axis=1)
    if np.any(valid):
        rows = flat[valid]
        nonzero = np.linalg.norm(rows, axis=1) > 0
        if np.any(nonzero):
            sim = cosine_similarity(rows[nonzero], ref.reshape(1, -1)).ravel()
            valid_idx = np.flatnonzero(valid)
            sim_flat[valid_idx[nonzero]] = sim.astype(np.float32)

    return sim_flat.reshape(H, W)


timings: dict[str, float] = {}

ModuleNotFoundError: No module named 'sklearn'

---
## Load the AEF collection

The AEF collection is prebuilt and hosted on Hugging Face as a Parquet
index. `rasteret.load()` downloads only the metadata shards — no pixels
yet. `subset()` filters lazily to Franklin County, 2024.

Also available on [Source Cooperative](https://source.coop/terrafloww/aef-v1-annual-rasteret) via S3:
`rasteret.load("s3://us-west-2.opendata.source.coop/terrafloww/aef-v1-annual-rasteret/")`

In [ ]:
collection = rasteret.load("aef/v1-annual")
collection.describe()

In [ ]:
COUNTY_URL = "https://www2.census.gov/geo/tiger/GENZ2021/shp/cb_2021_us_county_500k.zip"
counties = gpd.read_file(COUNTY_URL)
franklin = counties[counties["GEOID"] == "20059"].to_crs(4326)
franklin_geom = franklin.geometry.iloc[0]
franklin_utm = franklin.to_crs(32615)
county_geom_utm = franklin_utm.geometry.iloc[0]

sub = collection.subset(
    bbox=tuple(franklin.total_bounds.tolist()),
    date_range=("2024-01-01", "2024-12-31"),
)
print(f"{len(sub)} AEF tiles cover Franklin County in 2024")

---
## Reference embedding with `sample_points`

`sample_points` reads only the tiles containing the query points — no
full-AOI load. It returns a PyArrow table (Arrow-native, zero-copy).

We pivot the table with DuckDB SQL — no pandas needed — to get a
(3, 64) matrix, dequantize, and average into a single 64-dim reference
vector.

In [ ]:
t0 = time.perf_counter()

pts = [Point(lon, lat) for lon, lat in REFERENCE_POINTS]
samples = sub.sample_points(pts, bands=ALL_BANDS, geometry_crs=4326)

timings["ref_sample"] = time.perf_counter() - t0
print(f"sample_points: {samples.num_rows} rows in {timings['ref_sample']:.1f}s")

# Pivot with DuckDB — zero-copy from Arrow, no pandas
con = duckdb.connect()
pivot = con.execute(
    "PIVOT samples ON band USING first(value) GROUP BY point_index ORDER BY point_index"
).fetch_arrow_table()

# Extract (3, 64) matrix in band order, dequantize, average
raw = np.column_stack([pivot.column(b).to_numpy() for b in ALL_BANDS])
ref_embeddings = [dequantize(raw[i]) for i in range(len(raw))]
reference_vector = np.nanmean(ref_embeddings, axis=0)
ref_norm = float(np.linalg.norm(reference_vector))

for i, emb in enumerate(ref_embeddings):
    cos = float(np.dot(emb, reference_vector) / (np.linalg.norm(emb) * ref_norm))
    print(f"  ref {i}: similarity to mean = {cos:.3f}")

---
## Approach A — `get_xarray` + cosine similarity

`get_xarray` mosaics all overlapping tiles into one `xarray.Dataset`
with spatial coordinates and CRS metadata. Pixels outside the input
polygon are masked with nodata.

> **Memory note:** For AEF (64 bands), `get_xarray` materializes the full AOI mosaic. For Franklin County this can peak around **20-24 GB RAM** even with `int8` source bands.

We stack bands into a (64, H, W) cube and pass it to
`cosine_similarity_map`.

In [ ]:
t0 = time.perf_counter()

ds = sub.get_xarray(
    geometries=franklin_geom,
    bands=ALL_BANDS,
    geometry_crs=4326,
)
timings["A_load"] = time.perf_counter() - t0

H, W = len(ds.y), len(ds.x)
print(
    f"mosaic shape: ({H}, {W})  bands: {len(ALL_BANDS)}  load: {timings['A_load']:.1f}s"
)

In [ ]:
t0 = time.perf_counter()

# to_dataarray() stacks all bands into a single (band, y, x) DataArray — no manual loop
cube_a = ds[ALL_BANDS].to_dataarray(dim="band").isel(time=0, geometry=0).values
sim_a = cosine_similarity_map(cube_a, reference_vector)

timings["A_cosine"] = time.perf_counter() - t0
timings["A_total"] = timings["A_load"] + timings["A_cosine"]

# xarray y-axis is ascending (south to north) — flip for north-up display
sim_a = sim_a[::-1]
del ds, cube_a

fin_a = sim_a[np.isfinite(sim_a)]
print(
    f"similarity  min={fin_a.min():.4f}  mean={fin_a.mean():.4f}  "
    f"max={fin_a.max():.4f}  pixels={fin_a.size:,}"
)
print(
    f"timing  load={timings['A_load']:.1f}s  "
    f"cosine={timings['A_cosine']:.1f}s  "
    f"total={timings['A_total']:.1f}s"
)

---
## Approach B — `get_gdf` + vectorized cosine

`get_gdf` returns a GeoDataFrame with one row per (record, band). Each
row carries a 2-D NumPy array in the `data` column. Records keep their
native shapes — no mosaicking, no resampling. This handles ragged tiles
naturally.

> **Memory note:** For this Franklin County setup with AEF 64 bands, `get_gdf` is typically around **~3 GB peak RAM** (lower than full `get_xarray`, usually higher than chip-wise TorchGeo streaming).

We stack bands per record and compute cosine similarity via a single
matrix multiply.

In [ ]:
t0 = time.perf_counter()

gdf = sub.get_gdf(
    geometries=franklin_geom,
    bands=ALL_BANDS,
    geometry_crs=4326,
)
timings["B_load"] = time.perf_counter() - t0

n_records = gdf["record_id"].nunique()
print(
    f"get_gdf: {len(gdf)} rows  "
    f"({n_records} records x {len(ALL_BANDS)} bands)  "
    f"load: {timings['B_load']:.1f}s"
)

In [ ]:
t0 = time.perf_counter()

total_pixels_b = 0
for rid in gdf["record_id"].unique():
    rec = gdf[gdf["record_id"] == rid].sort_values("band")
    cube = np.stack([row["data"] for _, row in rec.iterrows()], axis=0)
    sim_b = cosine_similarity_map(cube, reference_vector)

    fin = sim_b[np.isfinite(sim_b)]
    total_pixels_b += fin.size
    H_r, W_r = sim_b.shape
    print(
        f"  record {rid}: ({H_r}, {W_r})  "
        f"min={fin.min():.4f}  mean={fin.mean():.4f}  max={fin.max():.4f}  "
        f"pixels={fin.size:,}"
    )

timings["B_cosine"] = time.perf_counter() - t0
timings["B_total"] = timings["B_load"] + timings["B_cosine"]
del gdf

print(f"\ntotal pixels: {total_pixels_b:,}")
print(
    f"timing  load={timings['B_load']:.1f}s  "
    f"cosine={timings['B_cosine']:.1f}s  "
    f"total={timings['B_total']:.1f}s"
)

---
## Approach C — TorchGeo streaming

Uses `GridGeoSampler` for chip-wise streaming and stitches chip predictions with
`notebooks/utils_stitching.py` so the notebook avoids manual row/col placement math.

> **Memory note:** In this Franklin County setup, TorchGeo chip-wise loading is typically much lower peak RAM (about **3-5 GB**) than full-AOI `get_xarray` materialization.

In [ ]:
from affine import Affine
from rasterio.features import geometry_mask
from torchgeo.samplers import GridGeoSampler

try:
    from notebooks.utils_stitching import stitch_prediction_tiles
except ImportError:
    from utils_stitching import stitch_prediction_tiles

CHIP_PX = 1024

t0 = time.perf_counter()

dataset = sub.to_torchgeo_dataset(bands=ALL_BANDS)
sampler = GridGeoSampler(dataset, size=CHIP_PX, stride=CHIP_PX, roi=county_geom_utm)

res_x, res_y = dataset.res
roi_xmin, roi_ymin, roi_xmax, roi_ymax = county_geom_utm.bounds
out_w = round((roi_xmax - roi_xmin) / res_x)
out_h = round((roi_ymax - roi_ymin) / res_y)
n_chips = len(sampler)

print(f"{n_chips} chips ({CHIP_PX}x{CHIP_PX} px)  output grid: ({out_h}, {out_w})")

tiles = []
skipped = 0
for i, query in enumerate(sampler):
    try:
        sample = dataset[query]
    except Exception:
        skipped += 1
        continue

    chip = sample["image"].numpy().astype(np.float32)
    chip_sim = cosine_similarity_map(chip, reference_vector)
    tiles.append({"prediction": chip_sim, "transform": sample["transform"].numpy()})

    elapsed = time.perf_counter() - t0
    print(f"  chip {i + 1}/{n_chips}  ({elapsed:.0f}s)", end="\r")

sim_c = stitch_prediction_tiles(
    tiles,
    roi_bounds=(roi_xmin, roi_ymin, roi_xmax, roi_ymax),
    res=(res_x, res_y),
    reducer="overwrite",
    out_shape=(out_h, out_w),
)

# Mask to county polygon for apples-to-apples comparison
out_transform = Affine(res_x, 0, roi_xmin, 0, -res_y, roi_ymax)
county_mask = geometry_mask(
    [county_geom_utm],
    out_shape=(out_h, out_w),
    transform=out_transform,
    invert=True,
)
sim_c[~county_mask] = np.nan

timings["C_total"] = time.perf_counter() - t0
dataset.close()

fin_c = sim_c[np.isfinite(sim_c)]
print(
    f"\nsimilarity  min={fin_c.min():.4f}  mean={fin_c.mean():.4f}  "
    f"max={fin_c.max():.4f}  pixels={fin_c.size:,}"
)
print(
    f"timing  total={timings['C_total']:.1f}s  ({n_chips} chips, "
    f"used={len(tiles)}, skipped={skipped})"
)

---
## Results

Approaches A and C both produce spatial similarity maps. B returns
per-record statistics (useful when tiles have different shapes).

All three agree on max similarity (~0.975) confirming they read
the same underlying data. Pixel counts differ by design:

- **A and B** mask to the county polygon — pixel count matches the
  county's geographic area at 10 m (~14.9 M pixels).
- **C** tiles the AOI with a fixed grid. Edge chips include zero-filled
  regions beyond tile coverage. We apply a county polygon mask
  afterwards to make the map comparable.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)

for ax, arr, label in [
    (axes[0], sim_a, f"A  get_xarray ({timings['A_total']:.0f}s)"),
    (axes[1], sim_c, f"C  TorchGeo streaming ({timings['C_total']:.0f}s)"),
]:
    im = ax.imshow(arr, cmap="RdYlGn", vmin=0.4, vmax=1.0)
    ax.set_title(label)
    ax.set_axis_off()

plt.colorbar(im, ax=axes, shrink=0.7, label="cosine similarity")
plt.suptitle("Grain Silo Similarity — Franklin County, Kansas", fontsize=14)
plt.show()

### Vectorize high-similarity regions and visualize with folium

Use a plain folium map for publishing-friendly rendering in VSCode,
MkDocs, and GitHub notebook previews.

In [ ]:
from rasterio.features import shapes
from shapely.geometry import shape

THRESHOLD = 0.95

minx, miny, maxx, maxy = franklin_utm.total_bounds
dx = (maxx - minx) / sim_a.shape[1]
dy = (maxy - miny) / sim_a.shape[0]
# sim_a is north-up (row 0 = north), so origin is top-left = (minx, maxy)
transform_a = Affine(dx, 0, minx, 0, -dy, maxy)

hot = np.isfinite(sim_a) & (sim_a >= THRESHOLD)
polys = [
    shape(geom)
    for geom, val in shapes(hot.astype(np.uint8), mask=hot, transform=transform_a)
    if val == 1
]
matches = gpd.GeoDataFrame(
    {"match_id": range(len(polys))}, geometry=polys, crs=32615
).to_crs(4326)
print(f"{len(matches)} regions with similarity >= {THRESHOLD}")

In [ ]:
import branca.colormap as cm
import folium

matches["area_m2"] = matches.to_crs(32615).area
vmin = float(matches["area_m2"].min()) if len(matches) else 0.0
vmax = float(matches["area_m2"].max()) if len(matches) else 1.0
if vmax <= vmin:
    vmax = vmin + 1.0
cmap = cm.linear.YlOrRd_09.scale(vmin, vmax)

center = franklin.geometry.iloc[0].centroid
m = folium.Map(
    location=[float(center.y), float(center.x)],
    zoom_start=9,
    tiles="CartoDB dark_matter",
)

folium.GeoJson(
    franklin.__geo_interface__,
    name="Franklin County",
    style_function=lambda _: {
        "color": "#9aa0a6",
        "weight": 2,
        "fillColor": "#9aa0a6",
        "fillOpacity": 0.05,
    },
).add_to(m)

for _, row in matches.iterrows():
    color = cmap(float(row["area_m2"]))
    folium.GeoJson(
        row.geometry.__geo_interface__,
        style_function=lambda _, color=color: {
            "color": color,
            "weight": 1.2,
            "fillColor": color,
            "fillOpacity": 0.65,
        },
    ).add_to(m)

cmap.caption = "Match region area (m²)"
cmap.add_to(m)
m

---
## Timing comparison

Timings below are measured in this run only. Use them as local run diagnostics,
not cross-project benchmarks.

In [ ]:
print(f"{'':32s} {'Load':>8s} {'Cosine':>8s} {'Total':>8s}")
print(f"{'---':32s} {'---':>8s} {'---':>8s} {'---':>8s}")
print(
    f"{'A  get_xarray + sklearn':32s} "
    f"{timings['A_load']:7.0f}s {timings['A_cosine']:7.0f}s "
    f"{timings['A_total']:7.0f}s"
)
print(
    f"{'B  get_gdf + sklearn':32s} "
    f"{timings['B_load']:7.0f}s {timings['B_cosine']:7.0f}s "
    f"{timings['B_total']:7.0f}s"
)
print(
    f"{'C  TorchGeo streaming':32s} "
    f"{'--':>8s} {'--':>8s} "
    f"{timings['C_total']:7.0f}s"
)

---
## Summary

| API | When to use | Memory (Franklin, AEF 64 bands) | Speed |
|---|---|---|---|
| `sample_points` | Build reference vectors from known coordinates | Tiny (MB-scale) | Fast |
| `get_xarray` | AOI-wide map-style analysis (single mosaic) | Highest (~20-24 GB peak) | Fastest for this county |
| `get_gdf` | Record-wise analysis with explicit per-record arrays | Medium (~3 GB peak) | Fast |
| `to_torchgeo_dataset` | Streaming chips for training/inference loops | Lowest (~3-5 GB peak) | Slower, bounded-memory |

This notebook uses `sklearn.metrics.pairwise.cosine_similarity` for readability,
while showing three Rasteret access patterns (`sample_points`, `get_xarray`, `get_gdf`)
and TorchGeo streaming with a reusable stitch helper.
